# FitZone — Análisis Predictivo de Deserción (Churn) y Segmentación K-Means
**Asignatura:** Minería de Datos II · **VIII Semestre — Ingeniería de Sistemas**  
**Institución:** Fundación Universitaria Tecnológico Comfenalco

Este notebook documenta el flujo completo de ciencia de datos en **Python (`scikit-learn`)** integrado en producción con el backend Spring Boot y la app móvil Flutter de **FitZone**:
1. Exploración y carga del dataset (`dataset_fitzone_churn.csv`).
2. Preprocesamiento con `ColumnTransformer` (`StandardScaler` + `OneHotEncoder`).
3. Entrenamiento, validación cruzada (5-Fold StratifiedKFold) y comparación de **Random Forest**, **Gradient Boosting** y **Red Neuronal (MLP)**.
4. Visualización de métricas: **Barras comparativas**, **Curvas ROC-AUC**, **Matriz de Confusión** e **Importancia de Variables**.
5. Segmentación no supervisada con **K-Means (k=3)** y **Silhouette Score**.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, silhouette_score
)

plt.style.use('dark_background')
df = pd.read_csv('dataset_fitzone_churn.csv')
print(f'Total registros: {len(df)} | Columnas: {len(df.columns)}')
print('Distribución de cancelación (0=Permanece, 1=Cancela):')
print(df['cancelara'].value_counts(normalize=True).mul(100).round(2))
df.head()

## 1. Preprocesamiento y Entrenamiento Comparativo (3 Modelos Supervisados)

In [ ]:
NUMERIC_FEATURES = [
    'edad', 'frecuencia_semanal', 'tiempo_membresia_meses', 'pagos_atrasados',
    'satisfaccion_servicio', 'peso_actual', 'cambio_peso_mensual', 'ingresos_mensuales'
]
CATEGORICAL_FEATURES = [
    'genero', 'objetivo', 'plan_membresia', 'ha_asistido_ultima_semana',
    'usa_entrenador_personal', 'asiste_clases_grupales', 'lesiones_recientes', 'tiene_descuento'
]

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df['cancelara']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
])

models = {
    'Random Forest': RandomForestClassifier(n_estimators=180, max_depth=8, min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=140, learning_rate=0.07, max_depth=3, random_state=42),
    'Red Neuronal (MLP)': MLPClassifier(hidden_layer_sizes=(48, 24), activation='relu', alpha=0.01, max_iter=450, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
fitted = {}

for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fitted[name] = (pipe, y_pred, y_proba)
    results.append({
        'Modelo': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4),
        'CV-F1 (5-fold)': round(cv_f1.mean(), 4),
    })

res_df = pd.DataFrame(results)
res_df

## 2. Gráficas de Evaluación: Comparativa, Curvas ROC-AUC y Matriz de Confusión

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Comparativa de métricas
metrics_cols = ['Accuracy', 'F1-Score', 'ROC-AUC']
x = np.arange(len(res_df))
w = 0.25
colors = ['#FF2A4B', '#F5A623', '#38BDF8']
for i, col in enumerate(metrics_cols):
    axes[0].bar(x + i*w, res_df[col]*100, width=w, label=col, color=colors[i])
axes[0].set_xticks(x + w)
axes[0].set_xticklabels(res_df['Modelo'], rotation=12)
axes[0].set_ylim(70, 100)
axes[0].set_ylabel('Porcentaje (%)')
axes[0].set_title('Comparativa de Modelos ML')
axes[0].legend()

# 2) Curvas ROC
for (name, (_, _, y_proba)), c in zip(fitted.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_v = roc_auc_score(y_test, y_proba)
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc_v:.3f})', linewidth=2.2, color=c)
axes[1].plot([0, 1], [0, 1], '--', color='gray', alpha=0.6)
axes[1].set_xlabel('Falsos Positivos (FPR)')
axes[1].set_ylabel('Verdaderos Positivos (TPR)')
axes[1].set_title('Curvas ROC-AUC')
axes[1].legend(loc='lower right')

# 3) Matriz de confusión del modelo ganador (Random Forest)
_, rf_pred, _ = fitted['Random Forest']
cm = confusion_matrix(y_test, rf_pred)
im = axes[2].imshow(cm, cmap='Reds')
axes[2].set_xticks([0, 1]); axes[2].set_yticks([0, 1])
axes[2].set_xticklabels(['Permanece', 'Cancela'])
axes[2].set_yticklabels(['Permanece', 'Cancela'])
axes[2].set_xlabel('Predicción'); axes[2].set_ylabel('Real')
axes[2].set_title('Matriz de Confusión — Random Forest')
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=15, fontweight='bold', color='white')

plt.tight_layout()
plt.show()

## 3. Importancia de Variables y Segmentación No Supervisada (K-Means)

In [ ]:
rf_pipe = fitted['Random Forest'][0]
feat_names = [f.replace('num__', '').replace('cat__', '') for f in rf_pipe.named_steps['preprocessor'].get_feature_names_out()]
importances = rf_pipe.named_steps['classifier'].feature_importances_
top_idx = np.argsort(importances)[-8:]

CLUSTER_FEATURES = ['frecuencia_semanal', 'satisfaccion_servicio', 'tiempo_membresia_meses', 'pagos_atrasados']
scaler = StandardScaler()
X_cl = scaler.fit_transform(df[CLUSTER_FEATURES])
km = KMeans(n_clusters=3, n_init=15, random_state=42)
labels = km.fit_predict(X_cl)
sil = silhouette_score(X_cl, labels)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Importancia de variables
axes[0].barh([feat_names[i] for i in top_idx], importances[top_idx]*100, color='#FF2A4B')
axes[0].set_xlabel('Importancia (%)')
axes[0].set_title('Top 8 Variables Predictoras de Deserción')

# Dispersión K-Means (Frecuencia vs Satisfacción)
scatter = axes[1].scatter(
    df['frecuencia_semanal'], df['satisfaccion_servicio'],
    c=labels, cmap='coolwarm', alpha=0.75, edgecolors='white', linewidths=0.4
)
axes[1].set_xlabel('Frecuencia semanal (días)')
axes[1].set_ylabel('Satisfacción con el servicio (1-10)')
axes[1].set_title(f'Segmentación K-Means (k=3) · Silhouette={sil:.4f}')
plt.tight_layout()
plt.show()